# Notebook 4: Interactive Dash App

**Survey of Consumer Finances — Customer Segmentation Dashboard**

This notebook builds a full interactive web application using **Dash** (by Plotly). Users can:
- Select whether to use trimmed or raw variance for feature selection
- Choose the number of clusters K with a slider
- View live-updating cluster metrics (inertia, silhouette score)
- Explore PCA scatter plots and bar charts of cluster profiles

The final cell exports `app.py` for deployment to **Hugging Face Spaces**.

## 0. Colab Setup

Run this cell first when working in Google Colab. It installs Dash (with Jupyter support) and mounts your Google Drive.

> **Before running:** Upload the `customer-segmentation/` folder to your Google Drive (or clone your GitHub repo).
>
> The Dash app runs **inline inside the notebook** in Colab using `jupyter-dash`.

In [ ]:
# ── Clone the GitHub repository ────────────────────────────────────────────
# Replace the URL below with your actual GitHub repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'

!git clone {REPO_URL}

# ── Set working directory to the project folder ────────────────────────────
import os

# Replace YOUR_REPO_NAME with the actual cloned folder name
os.chdir('/content/YOUR_REPO_NAME/customer-segmentation')

# ── Install required packages ──────────────────────────────────────────────
!pip install pandas numpy plotly scikit-learn scipy dash jupyter-dash --quiet

print(f'Working directory: {os.getcwd()}')
print(f'Data file exists: {os.path.exists("data/SCF_2019.csv.gz")}')

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import mstats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output
import warnings

warnings.filterwarnings('ignore')

print('Libraries loaded!')

## 2. Wrangle Function

In [ ]:
def wrangle(filepath='data/SCF_2019.csv.gz'):
    """
    Load SCF 2019 data.
    - Implicate 1 (every 5th row)
    - TURNFEAR == 1
    - NETWORTH < 2,000,000
    """
    df = pd.read_csv(filepath, compression='gzip', index_col=0)
    df = df.iloc[::5].reset_index(drop=True)
    df = df[df['TURNFEAR'] == 1].copy()
    df = df[df['NETWORTH'] < 2_000_000].copy()
    return df.reset_index(drop=True)


# Load data once at startup
df = wrangle()
print(f'Data loaded: {df.shape[0]:,} households')

## 3. Feature Selection Helper

In [ ]:
def trimmed_variance(x, proportiontocut=0.1):
    """Variance after trimming extreme values."""
    x_trimmed = mstats.trimr(np.array(x), proportiontocut, proportiontocut)
    return np.var(x_trimmed.compressed())


def get_high_var_features(df, trimmed=True, n=5):
    """
    Return the n features with highest variance from the DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
    trimmed : bool
        If True, use trimmed variance. If False, use raw variance.
    n : int
        Number of top features to return.
    
    Returns
    -------
    list of str : column names of top n high-variance features
    """
    # Exclude non-feature columns
    exclude = ['YY1', 'Y1', 'WGT', 'TURNFEAR', 'RACECL4', 'RACECL', 'HHSEX',
               'MARRIED', 'KIDS', 'OCCAT1', 'OCCAT2', 'EDUC', 'INCOME_RANK']
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    feature_cols = [c for c in numeric_cols if c not in exclude]
    
    if trimmed:
        var_dict = {
            col: trimmed_variance(df[col].dropna().values)
            for col in feature_cols
            if df[col].notna().sum() > 50
        }
        var_series = pd.Series(var_dict).sort_values(ascending=False)
    else:
        var_series = df[feature_cols].var().sort_values(ascending=False)
    
    return var_series.head(n).index.tolist()


# Test the function
print('Top 5 features (trimmed):', get_high_var_features(df, trimmed=True))
print('Top 5 features (raw):    ', get_high_var_features(df, trimmed=False))

## 4. Bar Chart: High-Variance Features

In [ ]:
def serve_bar_chart(trimmed=True):
    """
    Return a Plotly bar chart of the 5 highest-variance features.
    
    Parameters
    ----------
    trimmed : bool
        Whether to use trimmed variance.
    
    Returns
    -------
    plotly.graph_objects.Figure
    """
    exclude = ['YY1', 'Y1', 'WGT', 'TURNFEAR', 'RACECL4', 'RACECL', 'HHSEX',
               'MARRIED', 'KIDS', 'OCCAT1', 'OCCAT2', 'EDUC', 'INCOME_RANK']
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    feature_cols = [c for c in numeric_cols if c not in exclude]
    
    if trimmed:
        var_dict = {
            col: trimmed_variance(df[col].dropna().values)
            for col in feature_cols
            if df[col].notna().sum() > 50
        }
        var_series = pd.Series(var_dict).sort_values(ascending=False).head(5)
        title_suffix = '(Trimmed Variance)'
    else:
        var_series = df[feature_cols].var().sort_values(ascending=False).head(5)
        title_suffix = '(Raw Variance)'
    
    fig = px.bar(
        x=var_series.values,
        y=var_series.index,
        orientation='h',
        title=f'Top 5 High-Variance Features {title_suffix}',
        labels={'x': 'Variance', 'y': 'Feature'},
        color=var_series.values,
        color_continuous_scale='Blues'
    )
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    return fig


# Test
serve_bar_chart(trimmed=True).show()

## 5. Model Metrics Helper

In [ ]:
def get_model_metrics(trimmed=True, k=3):
    """
    Build, train, and evaluate a KMeans pipeline.
    
    Parameters
    ----------
    trimmed : bool
        Whether to use trimmed variance for feature selection.
    k : int
        Number of clusters.
    
    Returns
    -------
    dict with keys:
        'model': fitted sklearn Pipeline
        'labels': np.ndarray of cluster labels
        'inertia': float
        'silhouette': float
        'X': pd.DataFrame of features used
        'X_scaled': np.ndarray of scaled features
    """
    high_var_cols = get_high_var_features(df, trimmed=trimmed)
    X = df[high_var_cols].copy()
    
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42, n_init=10)
    )
    model.fit(X)
    
    labels = model.named_steps['kmeans'].labels_
    inertia = model.named_steps['kmeans'].inertia_
    X_scaled = model.named_steps['standardscaler'].transform(X)
    sil = silhouette_score(X_scaled, labels)
    
    return {
        'model': model,
        'labels': labels,
        'inertia': inertia,
        'silhouette': sil,
        'X': X,
        'X_scaled': X_scaled
    }


# Test
metrics = get_model_metrics(trimmed=True, k=3)
print(f'Inertia: {metrics["inertia"]:,.2f}')
print(f'Silhouette Score: {metrics["silhouette"]:.4f}')

## 6. Metrics Display Helper

In [ ]:
def serve_metrics(trimmed=True, k=3):
    """
    Return Dash HTML components showing model metrics.
    
    Returns
    -------
    list of Dash components [H3 inertia, H3 silhouette]
    """
    result = get_model_metrics(trimmed=trimmed, k=k)
    return [
        html.H3(f'Inertia: {result["inertia"]:,.2f}',
                style={'color': '#636EFA', 'marginBottom': '8px'}),
        html.H3(f'Silhouette Score: {result["silhouette"]:.4f}',
                style={'color': '#00CC96'})
    ]


print('serve_metrics function defined.')

## 7. PCA Scatter Plot Helper

In [ ]:
def get_pca_labels(trimmed=True, k=3):
    """
    Get PCA-reduced 2D data with cluster labels.
    
    Parameters
    ----------
    trimmed : bool
    k : int
    
    Returns
    -------
    pd.DataFrame with columns ['PC1', 'PC2', 'labels']
    """
    result = get_model_metrics(trimmed=trimmed, k=k)
    
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(result['X_scaled'])
    
    df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
    df_pca['labels'] = result['labels'].astype(str)
    
    return df_pca


def serve_scatter_plot(trimmed=True, k=3):
    """
    Return a Plotly scatter plot of PCA-reduced clusters.
    
    Returns
    -------
    plotly.graph_objects.Figure
    """
    df_pca = get_pca_labels(trimmed=trimmed, k=k)
    
    fig = px.scatter(
        df_pca,
        x='PC1',
        y='PC2',
        color='labels',
        title=f'PCA Representation of Clusters (K={k})',
        labels={'PC1': 'PC1', 'PC2': 'PC2', 'labels': 'Cluster'},
        opacity=0.65,
        color_discrete_sequence=px.colors.qualitative.Plotly
    )
    fig.update_layout(legend_title_text='Cluster')
    return fig


# Test
serve_scatter_plot(trimmed=True, k=3).show()

## 8. Build the Dash Application

In [ ]:
# Instantiate the Dash app using JupyterDash for Colab compatibility
app = JupyterDash(__name__)

# ─────────────────────────────────────────────
# App Layout
# ─────────────────────────────────────────────
app.layout = html.Div(
    style={'fontFamily': 'Arial, sans-serif', 'maxWidth': '1200px',
           'margin': '0 auto', 'padding': '24px'},
    children=[
        # ── Header ──────────────────────────────
        html.H1(
            'Survey of Consumer Finances',
            style={'textAlign': 'center', 'color': '#1a1a2e', 'marginBottom': '4px'}
        ),
        html.P(
            'Customer Segmentation Dashboard — Households turned down or fearing credit denial (TURNFEAR=1, NetWorth < $2M)',
            style={'textAlign': 'center', 'color': '#555', 'marginBottom': '32px'}
        ),

        # ── Section 1: High Variance Features ───
        html.H2('High Variance Features', style={'color': '#16213e'}),
        html.P('Choose whether to compute variance on raw data or with trimmed extremes:'),
        dcc.RadioItems(
            id='trim-button',
            options=[
                {'label': '  Trimmed Variance', 'value': True},
                {'label': '  Raw Variance (not trimmed)', 'value': False}
            ],
            value=True,
            inline=True,
            style={'marginBottom': '12px'}
        ),
        dcc.Graph(id='bar-chart'),

        html.Hr(),

        # ── Section 2: K-Means Clustering ───────
        html.H2('K-means Clustering', style={'color': '#16213e'}),
        html.H3('Number of Clusters (k)'),
        dcc.Slider(
            id='k-slider',
            min=2,
            max=12,
            step=1,
            value=3,
            marks={i: str(i) for i in range(2, 13)},
            tooltip={'placement': 'bottom', 'always_visible': True}
        ),

        html.Br(),

        # ── Metrics ──────────────────────────────
        html.Div(
            id='metrics',
            style={'background': '#f0f4ff', 'borderRadius': '8px',
                   'padding': '16px', 'marginBottom': '24px'}
        ),

        # ── PCA Scatter ──────────────────────────
        dcc.Graph(id='pca-scatter'),

        html.Hr(),

        # ── Footer ───────────────────────────────
        html.P(
            'Data: Federal Reserve Board — Survey of Consumer Finances 2019. '
            'Methodology: K-Means clustering with StandardScaler preprocessing + PCA visualization.',
            style={'color': '#999', 'fontSize': '12px', 'textAlign': 'center'}
        )
    ]
)

print('Layout defined!')

## 9. Callbacks

In [ ]:
# ─── Callback 1: Update bar chart based on trimmed toggle ───
@app.callback(
    Output('bar-chart', 'figure'),
    Input('trim-button', 'value')
)
def update_bar_chart(trimmed):
    return serve_bar_chart(trimmed=trimmed)


# ─── Callback 2: Update metrics based on trim + k ───────────
@app.callback(
    Output('metrics', 'children'),
    Input('trim-button', 'value'),
    Input('k-slider', 'value')
)
def update_metrics(trimmed, k):
    return serve_metrics(trimmed=trimmed, k=k)


# ─── Callback 3: Update PCA scatter based on trim + k ───────
@app.callback(
    Output('pca-scatter', 'figure'),
    Input('trim-button', 'value'),
    Input('k-slider', 'value')
)
def update_scatter_plot(trimmed, k):
    return serve_scatter_plot(trimmed=trimmed, k=k)


print('Callbacks registered!')

## 10. Run the App

Run the cell below to launch the Dash app **inline in Colab**. The dashboard will appear directly below the cell.

- `mode='inline'` — renders inside the notebook output
- `mode='external'` — opens in a separate browser tab (use if inline doesn't work)
- `mode='jupyterlab'` — use inside JupyterLab

In [ ]:
# Run inline in Colab (change mode='external' to open in a new tab)
app.run_server(mode='inline', port=8050, height=900)

## 11. Export `app.py` for Hugging Face Spaces Deployment

In [ ]:
app_py_code = '''
# app.py — Customer Segmentation Dashboard
# Deploy this file to Hugging Face Spaces (Dash type)
# Requires: requirements.txt with dash, pandas, numpy, scipy, scikit-learn, plotly

import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import mstats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from dash import Dash, dcc, html, Input, Output
import warnings

warnings.filterwarnings("ignore")


def trimmed_variance(x, proportiontocut=0.1):
    x_trimmed = mstats.trimr(np.array(x), proportiontocut, proportiontocut)
    return np.var(x_trimmed.compressed())


def wrangle(filepath="data/SCF_2019.csv.gz"):
    df = pd.read_csv(filepath, compression="gzip", index_col=0)
    df = df.iloc[::5].reset_index(drop=True)
    df = df[df["TURNFEAR"] == 1].copy()
    df = df[df["NETWORTH"] < 2_000_000].copy()
    return df.reset_index(drop=True)


df = wrangle()

EXCLUDE = ["YY1", "Y1", "WGT", "TURNFEAR", "RACECL4", "RACECL", "HHSEX",
           "MARRIED", "KIDS", "OCCAT1", "OCCAT2", "EDUC", "INCOME_RANK"]


def get_high_var_features(df, trimmed=True, n=5):
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    feature_cols = [c for c in numeric_cols if c not in EXCLUDE]
    if trimmed:
        var_dict = {col: trimmed_variance(df[col].dropna().values)
                    for col in feature_cols if df[col].notna().sum() > 50}
        var_series = pd.Series(var_dict).sort_values(ascending=False)
    else:
        var_series = df[feature_cols].var().sort_values(ascending=False)
    return var_series.head(n).index.tolist()


def serve_bar_chart(trimmed=True):
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    feature_cols = [c for c in numeric_cols if c not in EXCLUDE]
    if trimmed:
        var_dict = {col: trimmed_variance(df[col].dropna().values)
                    for col in feature_cols if df[col].notna().sum() > 50}
        var_series = pd.Series(var_dict).sort_values(ascending=False).head(5)
        title_suffix = "(Trimmed Variance)"
    else:
        var_series = df[feature_cols].var().sort_values(ascending=False).head(5)
        title_suffix = "(Raw Variance)"
    fig = px.bar(x=var_series.values, y=var_series.index, orientation="h",
                 title=f"Top 5 High-Variance Features {title_suffix}",
                 labels={"x": "Variance", "y": "Feature"},
                 color=var_series.values, color_continuous_scale="Blues")
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    return fig


def get_model_metrics(trimmed=True, k=3):
    high_var_cols = get_high_var_features(df, trimmed=trimmed)
    X = df[high_var_cols].copy()
    model = make_pipeline(StandardScaler(), KMeans(n_clusters=k, random_state=42, n_init=10))
    model.fit(X)
    labels = model.named_steps["kmeans"].labels_
    inertia = model.named_steps["kmeans"].inertia_
    X_scaled = model.named_steps["standardscaler"].transform(X)
    sil = silhouette_score(X_scaled, labels)
    return {"model": model, "labels": labels, "inertia": inertia,
            "silhouette": sil, "X": X, "X_scaled": X_scaled}


def serve_metrics(trimmed=True, k=3):
    result = get_model_metrics(trimmed=trimmed, k=k)
    return [
        html.H3(f"Inertia: {result['inertia']:,.2f}", style={"color": "#636EFA"}),
        html.H3(f"Silhouette Score: {result['silhouette']:.4f}", style={"color": "#00CC96"})
    ]


def get_pca_labels(trimmed=True, k=3):
    result = get_model_metrics(trimmed=trimmed, k=k)
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(result["X_scaled"])
    df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
    df_pca["labels"] = result["labels"].astype(str)
    return df_pca


def serve_scatter_plot(trimmed=True, k=3):
    df_pca = get_pca_labels(trimmed=trimmed, k=k)
    fig = px.scatter(df_pca, x="PC1", y="PC2", color="labels",
                     title=f"PCA Representation of Clusters (K={k})",
                     labels={"PC1": "PC1", "PC2": "PC2", "labels": "Cluster"},
                     opacity=0.65,
                     color_discrete_sequence=px.colors.qualitative.Plotly)
    return fig


app = Dash(__name__)
server = app.server  # expose Flask server for Hugging Face

app.layout = html.Div(
    style={"fontFamily": "Arial, sans-serif", "maxWidth": "1200px",
           "margin": "0 auto", "padding": "24px"},
    children=[
        html.H1("Survey of Consumer Finances",
                style={"textAlign": "center", "color": "#1a1a2e"}),
        html.P("Customer Segmentation — Households turned down or fearing credit denial",
               style={"textAlign": "center", "color": "#555", "marginBottom": "32px"}),
        html.H2("High Variance Features"),
        dcc.RadioItems(id="trim-button",
                       options=[{"label": " Trimmed", "value": True},
                                {"label": " Not Trimmed", "value": False}],
                       value=True, inline=True),
        dcc.Graph(id="bar-chart"),
        html.Hr(),
        html.H2("K-means Clustering"),
        html.H3("Number of Clusters (k)"),
        dcc.Slider(id="k-slider", min=2, max=12, step=1, value=3,
                   marks={i: str(i) for i in range(2, 13)}),
        html.Br(),
        html.Div(id="metrics",
                 style={"background": "#f0f4ff", "borderRadius": "8px",
                        "padding": "16px", "marginBottom": "24px"}),
        dcc.Graph(id="pca-scatter"),
    ]
)


@app.callback(Output("bar-chart", "figure"), Input("trim-button", "value"))
def update_bar_chart(trimmed):
    return serve_bar_chart(trimmed=trimmed)


@app.callback(Output("metrics", "children"),
              Input("trim-button", "value"), Input("k-slider", "value"))
def update_metrics(trimmed, k):
    return serve_metrics(trimmed=trimmed, k=k)


@app.callback(Output("pca-scatter", "figure"),
              Input("trim-button", "value"), Input("k-slider", "value"))
def update_scatter_plot(trimmed, k):
    return serve_scatter_plot(trimmed=trimmed, k=k)


if __name__ == "__main__":
    app.run(debug=False, host="0.0.0.0", port=7860)
'''

with open('app.py', 'w') as f:
    f.write(app_py_code.strip())

print('app.py written successfully!')
print('Upload app.py + requirements.txt + data/ to Hugging Face Spaces.')